<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo"  />
    </a>
</p>


# **Data Wrangling Lab**


Estimated time needed: **45** minutes


In this lab, you will perform data wrangling tasks to prepare raw data for analysis. Data wrangling involves cleaning, transforming, and organizing data into a structured format suitable for analysis. This lab focuses on tasks like identifying inconsistencies, encoding categorical variables, and feature transformation.


## Objectives


After completing this lab, you will be able to:


- Identify and remove inconsistent data entries.

- Encode categorical variables for analysis.

- Handle missing values using multiple imputation strategies.

- Apply feature scaling and transformation techniques.


#### Intsall the required libraries


In [1]:
# The original `!pip install` magic commands only run in Jupyter environments.
# They are kept below as comments. Use subprocess instead so the packages are
# installed with the same Python interpreter that is selected in VS Code.
# !pip install pandas
# !pip install matplotlib
import importlib.util
import subprocess
import sys

def install(package, import_name=None):
    """Quietly pip-install a package if it is not already available."""
    import_name = import_name or package
    try:
        spec = importlib.util.find_spec(import_name)
    except (ImportError, ModuleNotFoundError, ValueError):
        spec = None
    if spec is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', package])

install('pandas', 'pandas')
install('matplotlib', 'matplotlib')

## Tasks


#### Step 1: Import the necessary module.


### 1. Load the Dataset


<h5>1.1 Import necessary libraries and load the dataset.</h5>


Ensure the dataset is loaded correctly by displaying the first few rows.


In [2]:
# Import necessary libraries
import os
import urllib.request
import urllib.error
import pandas as pd

# Load the Stack Overflow survey data
# (cached locally so the notebook still runs when the network is down)
dataset_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/n01PQ9pSmiRX6520flujwQ/survey-data.csv"
local_file = "survey-data.csv"

def load_dataset(url, local_path, retries=3):
    # 1) Prefer a previously downloaded local copy (works offline)
    if os.path.exists(local_path) and os.path.getsize(local_path) > 0:
        try:
            return pd.read_csv(local_path)
        except Exception:
            pass  # fall through to re-download
    # 2) Otherwise download with a few retries
    last_err = None
    for attempt in range(1, retries + 1):
        try:
            urllib.request.urlretrieve(url, local_path)
            return pd.read_csv(local_path)
        except (urllib.error.URLError, OSError) as e:
            last_err = e
            print(f"Download attempt {attempt}/{retries} failed: {e}")
    raise RuntimeError(
        "Could not download the dataset.\n"
        f"URL: {url}\nLast error: {last_err}\n"
        "Check your internet connection / VPN / proxy, or place a copy of the "
        f"CSV named '{local_path}' next to this notebook."
    )

df = load_dataset(dataset_url, local_file)

# Display the first few rows
print(df.head())

   ResponseId                      MainBranch                 Age  \
0           1  I am a developer by profession  Under 18 years old   
1           2  I am a developer by profession     35-44 years old   
2           3  I am a developer by profession     45-54 years old   
3           4           I am learning to code     18-24 years old   
4           5  I am a developer by profession     18-24 years old   

            Employment RemoteWork   Check  \
0  Employed, full-time     Remote  Apples   
1  Employed, full-time     Remote  Apples   
2  Employed, full-time     Remote  Apples   
3   Student, full-time        NaN  Apples   
4   Student, full-time        NaN  Apples   

                                    CodingActivities  \
0                                              Hobby   
1  Hobby;Contribute to open-source projects;Other...   
2  Hobby;Contribute to open-source projects;Other...   
3                                                NaN   
4                                 

#### 2. Explore the Dataset


<h5>2.1 Summarize the dataset by displaying the column data types, counts, and missing values.</h5>


In [3]:
# Write your code here
# Display column data types, non-null counts, and memory usage
df.info()

# Number of missing values per column
print("\nMissing values per column (top 10):")
print(df.isna().sum().sort_values(ascending=False).head(10))

<class 'pandas.DataFrame'>
RangeIndex: 65437 entries, 0 to 65436
Columns: 114 entries, ResponseId to JobSat
dtypes: float64(13), int64(1), str(100)
memory usage: 56.9 MB

Missing values per column (top 10):
AINextMuch less integrated       64289
AINextLess integrated            63082
AINextNo change                  52939
AINextMuch more integrated       51999
EmbeddedAdmired                  48704
EmbeddedWantToWorkWith           47837
EmbeddedHaveWorkedWith           43223
ConvertedCompYearly              42002
AIToolNot interested in Using    41023
AINextMore integrated            41009
dtype: int64


<h5>2.2 Generate basic statistics for numerical columns.</h5>


In [4]:
# Write your code here
# Basic statistics for numerical columns
print(df.describe())

         ResponseId      CompTotal       WorkExp  JobSatPoints_1  \
count  65437.000000   3.374000e+04  29658.000000    29324.000000   
mean   32719.000000  2.963841e+145     11.466957       18.581094   
std    18890.179119  5.444117e+147      9.168709       25.966221   
min        1.000000   0.000000e+00      0.000000        0.000000   
25%    16360.000000   6.000000e+04      4.000000        0.000000   
50%    32719.000000   1.100000e+05      9.000000       10.000000   
75%    49078.000000   2.500000e+05     16.000000       22.000000   
max    65437.000000  1.000000e+150     50.000000      100.000000   

       JobSatPoints_4  JobSatPoints_5  JobSatPoints_6  JobSatPoints_7  \
count    29393.000000    29411.000000    29450.000000     29448.00000   
mean         7.522140       10.060857       24.343232        22.96522   
std         18.422661       21.833836       27.089360        27.01774   
min          0.000000        0.000000        0.000000         0.00000   
25%          0.000000 

### 3. Identifying and Removing Inconsistencies


<h5>3.1 Identify inconsistent or irrelevant entries in specific columns (e.g., Country).</h5>


In [5]:
# Write your code here
# Inspect the Country column for inconsistent/irrelevant entries
print("Number of unique countries:", df['Country'].nunique())
print("\nCountry value counts (top 20):")
print(df['Country'].value_counts().head(20))

# Flag very rare entries (likely typos or 'Other' buckets) and long-form names
rare = df['Country'].value_counts()
rare = rare[rare < 5]
print("\nCountries with fewer than 5 respondents (", len(rare), "):")
print(rare)

Number of unique countries: 185

Country value counts (top 20):
Country
United States of America                                11095
Germany                                                  4947
India                                                    4231
United Kingdom of Great Britain and Northern Ireland     3224
Ukraine                                                  2672
France                                                   2110
Canada                                                   2104
Poland                                                   1534
Netherlands                                              1449
Brazil                                                   1375
Italy                                                    1341
Australia                                                1260
Spain                                                    1123
Sweden                                                   1016
Russian Federation                                        92

<h5>3.2 Standardize entries in columns like Country or EdLevel by mapping inconsistent values to a consistent format.</h5>


In [6]:
## Write your code here
# Standardize inconsistent country names to a consistent format
country_mapping = {
    'United Kingdom of Great Britain and Northern Ireland': 'United Kingdom',
    'United States of America': 'United States',
}
df['Country'] = df['Country'].replace(country_mapping)

# EdLevel is already fairly consistent; print its values so any further
# mapping can be applied the same way if needed
print("EdLevel value counts:")
print(df['EdLevel'].value_counts())

print("\nCountry value counts after standardization (top 10):")
print(df['Country'].value_counts().head(10))

EdLevel value counts:
EdLevel
Bachelor’s degree (B.A., B.S., B.Eng., etc.)                                          24942
Master’s degree (M.A., M.S., M.Eng., MBA, etc.)                                       15557
Some college/university study without earning a degree                                 7651
Secondary school (e.g. American high school, German Realschule or Gymnasium, etc.)     5793
Professional degree (JD, MD, Ph.D, Ed.D, etc.)                                         2970
Associate degree (A.A., A.S., etc.)                                                    1793
Primary/elementary school                                                              1146
Something else                                                                          932
Name: count, dtype: int64

Country value counts after standardization (top 10):
Country
United States     11095
Germany            4947
India              4231
United Kingdom     3224
Ukraine            2672
France             2110
Ca

### 4. Encoding Categorical Variables


<h5>4.1 Encode the Employment column using one-hot encoding.</h5>


In [7]:
## Write your code here
# One-hot encode the Employment column
employment_dummies = pd.get_dummies(df['Employment'], prefix='Employment')
df = pd.concat([df, employment_dummies], axis=1)

print("Employment columns created:")
print([c for c in df.columns if c.startswith('Employment_')])
print("\nShape after encoding:", df.shape)

Employment columns created:
['Employment_Employed, full-time', 'Employment_Employed, full-time;Employed, part-time', 'Employment_Employed, full-time;Independent contractor, freelancer, or self-employed', 'Employment_Employed, full-time;Independent contractor, freelancer, or self-employed;Employed, part-time', 'Employment_Employed, full-time;Independent contractor, freelancer, or self-employed;Employed, part-time;Retired', 'Employment_Employed, full-time;Independent contractor, freelancer, or self-employed;Not employed, and not looking for work', 'Employment_Employed, full-time;Independent contractor, freelancer, or self-employed;Not employed, and not looking for work;Employed, part-time', 'Employment_Employed, full-time;Independent contractor, freelancer, or self-employed;Not employed, and not looking for work;Student, part-time', 'Employment_Employed, full-time;Independent contractor, freelancer, or self-employed;Retired', 'Employment_Employed, full-time;Independent contractor, freela

### 5. Handling Missing Values


<h5>5.1 Identify columns with the highest number of missing values.</h5>


In [8]:
## Write your code here
# Identify columns with the highest number of missing values
missing_values = df.isna().sum().sort_values(ascending=False)
print("Top 10 columns by missing values:")
print(missing_values.head(10))

Top 10 columns by missing values:
AINextMuch less integrated       64289
AINextLess integrated            63082
AINextNo change                  52939
AINextMuch more integrated       51999
EmbeddedAdmired                  48704
EmbeddedWantToWorkWith           47837
EmbeddedHaveWorkedWith           43223
ConvertedCompYearly              42002
AIToolNot interested in Using    41023
AINextMore integrated            41009
dtype: int64


<h5>5.2 Impute missing values in numerical columns (e.g., `ConvertedCompYearly`) with the mean or median.</h5>


In [9]:
## Write your code here
# Impute missing values in ConvertedCompYearly with the median
# (median is robust to the extreme outliers present in compensation data)
median_comp = df['ConvertedCompYearly'].median()
print("Median ConvertedCompYearly:", median_comp)
print("Missing before imputation:", df['ConvertedCompYearly'].isna().sum())
df['ConvertedCompYearly'] = df['ConvertedCompYearly'].fillna(median_comp)
print("Missing after imputation:", df['ConvertedCompYearly'].isna().sum())

Median ConvertedCompYearly: 65000.0
Missing before imputation: 42002
Missing after imputation: 0


<h5>5.3 Impute missing values in categorical columns (e.g., `RemoteWork`) with the most frequent value.</h5>


In [10]:
## Write your code here
# Impute missing values in RemoteWork with the most frequent value
most_frequent = df['RemoteWork'].mode()[0]
print("Most frequent RemoteWork value:", most_frequent)
print("Missing before imputation:", df['RemoteWork'].isna().sum())
df['RemoteWork'] = df['RemoteWork'].fillna(most_frequent)
print("Missing after imputation:", df['RemoteWork'].isna().sum())

Most frequent RemoteWork value: Hybrid (some remote, some in-person)
Missing before imputation: 10631
Missing after imputation: 0


### 6. Feature Scaling and Transformation


<h5>6.1 Apply Min-Max Scaling to normalize the `ConvertedCompYearly` column.</h5>


In [11]:
## Write your code here
# Apply Min-Max Scaling to normalize ConvertedCompYearly to a 0-1 range
col = 'ConvertedCompYearly'
min_val = df[col].min()
max_val = df[col].max()
df['ConvertedCompYearly_MinMax'] = (df[col] - min_val) / (max_val - min_val)
print("Min:", df['ConvertedCompYearly_MinMax'].min(),
      "Max:", df['ConvertedCompYearly_MinMax'].max())
print(df[['ConvertedCompYearly', 'ConvertedCompYearly_MinMax']].head())

Min: 0.0 Max: 1.0
   ConvertedCompYearly  ConvertedCompYearly_MinMax
0              65000.0                    0.003998
1              65000.0                    0.003998
2              65000.0                    0.003998
3              65000.0                    0.003998
4              65000.0                    0.003998


<h5>6.2 Log-transform the ConvertedCompYearly column to reduce skewness.</h5>


In [12]:
## Write your code here
# Log-transform ConvertedCompYearly to reduce skewness
import numpy as np
df['ConvertedCompYearly_Log'] = np.log1p(df['ConvertedCompYearly'])
print("Log-transformed column created.")
print(df[['ConvertedCompYearly', 'ConvertedCompYearly_Log']].describe())

Log-transformed column created.
       ConvertedCompYearly  ConvertedCompYearly_Log
count         6.543700e+04             65437.000000
mean          7.257636e+04                10.976053
std           1.122207e+05                 0.851456
min           1.000000e+00                 0.693147
25%           6.500000e+04                11.082158
50%           6.500000e+04                11.082158
75%           6.500000e+04                11.082158
max           1.625660e+07                16.604010


### 7. Feature Engineering


<h5>7.1 Create a new column `ExperienceLevel` based on the `YearsCodePro` column:</h5>


In [13]:
## Write your code here
# Create a new column 'ExperienceLevel' based on 'YearsCodePro'
def categorize_experience(years):
    if pd.isna(years):
        return 'Unknown'
    # handle the special string values present in this dataset
    if isinstance(years, str):
        if years == 'Less than 1 year':
            years = 0.5
        elif years == 'More than 50 years':
            years = 50
        else:
            try:
                years = float(years)
            except ValueError:
                return 'Unknown'
    years = float(years)
    if years < 5:
        return 'Junior'
    elif years < 10:
        return 'Mid-level'
    else:
        return 'Senior'

df['ExperienceLevel'] = df['YearsCodePro'].apply(categorize_experience)
print("ExperienceLevel value counts:")
print(df['ExperienceLevel'].value_counts())

ExperienceLevel value counts:
ExperienceLevel
Senior       21711
Junior       16971
Unknown      13827
Mid-level    12928
Name: count, dtype: int64


### Summary


In this lab, you:

- Explored the dataset to identify inconsistencies and missing values.

- Encoded categorical variables for analysis.

- Handled missing values using imputation techniques.

- Normalized and transformed numerical data to prepare it for analysis.

- Engineered a new feature to enhance data interpretation.


Copyright © IBM Corporation. All rights reserved.
